# Multilingual NER Model Fine-Tuning

This notebook handles the fine-tuning of the XLM-RoBERTa model using the interleaved Hungarian, English, and German datasets.

## Environment Setup

Initializing hardware settings and loading core libraries for training.

In [1]:
import os

import evaluate
import numpy as np
import torch
from datasets import load_from_disk
from dotenv import load_dotenv
from huggingface_hub import notebook_login
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    pipeline,
)
from transformers.utils.notebook import NotebookProgressCallback

import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

load_dotenv()

cuda


True

## Weights & Biases Logging

Connecting to W&B to track metrics during the experiment runs.

In [2]:
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: rchrdhlzhfr (rchrdhlzhfr-personal) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Loading Processed Datasets

Loading the tokenized and aligned datasets.

In [3]:
# Load interleaved and gold-only datasets
interleaved_dataset = load_from_disk("../data/processed/harmonized_dataset/interleaved_dataset")
tokenized_dataset = load_from_disk("../data/processed/tokenized_dataset/tokenized_dataset")

gold_only_dataset = load_from_disk("../data/processed/harmonized_dataset/gold_only_dataset")
gold_only_tokenized_dataset = load_from_disk("../data/processed/tokenized_dataset/gold_only_tokenized_dataset")

# Load individual language datasets (harmonized)
hun_dataset = load_from_disk("../data/processed/harmonized_dataset/hun_dataset")
eng_dataset = load_from_disk("../data/processed/harmonized_dataset/eng_dataset")
ger_dataset = load_from_disk("../data/processed/harmonized_dataset/ger_dataset")

# Load individual language datasets (tokenized)
tokenized_hun_dataset = load_from_disk("../data/processed/tokenized_dataset/hun_tokenized_dataset")
tokenized_eng_dataset = load_from_disk("../data/processed/tokenized_dataset/eng_tokenized_dataset")
tokenized_ger_dataset = load_from_disk("../data/processed/tokenized_dataset/ger_tokenized_dataset")

## Tokenizer, model and data collator initialization

In [4]:
model_id = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_id)
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

Preparing dictionaries for model initialization on interleaved dataset

In [5]:
label_names = interleaved_dataset['train']['ner'].features.feature.names
label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [6]:
id2label = {i: label for i, label in enumerate(label_names)}
label2id = {v: k for k, v in id2label.items()}

In [7]:
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Data Collator Verification

Ensuring labels are correctly padded and aligned within batches.

In [8]:
# Lengths of the first three labels before data collator
[print(len(tokenized_dataset['train']['labels'][i])) for i in range(3)];

67
84
35


In [9]:
example_batch = data_collator([tokenized_dataset["train"][i] for i in range(3)])
example_batch["labels"]

tensor([[-100,    0, -100, -100, -100,    0, -100,    0,    3, -100, -100,    4,
            4, -100, -100,    4,    0,    3, -100,    0,    0, -100,    0, -100,
            0,    3, -100,    4,    0, -100,    0,    0, -100, -100,    0,    0,
         -100, -100,    0,    0,    0, -100, -100, -100,    0,    0,    1,    2,
            0, -100,    0,    3, -100,    4,    0, -100, -100, -100, -100,    0,
            3, -100,    0, -100,    0, -100, -100, -100, -100, -100, -100, -100,
         -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100],
        [-100,    0,    0,    0,    0,    0, -100,    0,    0,    0,    0,    0,
            0,    0,    0, -100,    0,    0,    0,    0,    0,    0,    0,    0,
            0, -100, -100,    0, -100,    5,    0, -100,    0,    0,    0,    0,
            0,    0,    0, -100,    0, -100,    0, -100,    0,    0, -100,    0,
         -100,    0,    0,    0,    0,    0,    0,    0,    0, -100,    0,    0,
            0,    0, -100, 

Inspecting the first three labels after data collator

In [10]:
# Lengths of the first three labels after data collator
[print(len(example_batch['labels'][i])) for i in range(3)];

84
84
84


## Evaluation Metrics

Defining the `compute_metrics` function using `seqeval` to calculate Precision, Recall, and F1 score for all entity tags.

In [11]:
metric = evaluate.load("seqeval")

In [12]:
def compute_metrics(eval_preds, label_names):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    # Remove ignored index (special tokens) and convert to labels
    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [
        [label_names[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    results = {
        "overall_precision": all_metrics["overall_precision"],
        "overall_recall": all_metrics["overall_recall"],
        "overall_f1": all_metrics["overall_f1"],
        "overall_accuracy": all_metrics["overall_accuracy"],
    }

    for k, v in all_metrics.items():
        if k not in ["overall_precision", "overall_recall", "overall_f1", "overall_accuracy"]:
            results[f"{k}_f1"] = v["f1"]
            results[f"{k}_precision"] = v["precision"]
            results[f"{k}_recall"] = v["recall"]

    return results


## Training Configuration

Setting hyper-parameters such as learning rate, batch size, and weight decay.

In [13]:
# Login to HuggingFace
notebook_login()

In [14]:
args = TrainingArguments(
    output_dir="../xlm-roberta-ner-hun_eng_ger",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="eval_combined_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    per_device_train_batch_size=96,
    per_device_eval_batch_size=192,
    bf16=True,
    bf16_full_eval=True,
    dataloader_num_workers=8,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    warmup_ratio=0.1,
    push_to_hub=False,
    report_to="wandb",
    run_name="ner_exp_hun_eng_ger"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


## Trainer Setup & Training

Starting the training process with the Hugging Face `Trainer`.

In [15]:
eval_datasets = {
    "combined": tokenized_dataset["validation"],
    "hun": tokenized_hun_dataset["validation"],
    "eng": tokenized_eng_dataset["validation"],
    "ger": tokenized_ger_dataset["validation"]
}

In [16]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=eval_datasets,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, label_names=label_names),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [17]:
wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_eng_ger_training"
    )

trainer.train()

wandb.finish()

Epoch,Training Loss,Validation Loss,Combined Loss,Combined Overall Precision,Combined Overall Recall,Combined Overall F1,Combined Overall Accuracy,Combined Loc F1,Combined Loc Precision,Combined Loc Recall,Combined Misc F1,Combined Misc Precision,Combined Misc Recall,Combined Org F1,Combined Org Precision,Combined Org Recall,Combined Per F1,Combined Per Precision,Combined Per Recall,Hun Loss,Hun Overall Precision,Hun Overall Recall,Hun Overall F1,Hun Overall Accuracy,Hun Loc F1,Hun Loc Precision,Hun Loc Recall,Hun Misc F1,Hun Misc Precision,Hun Misc Recall,Hun Org F1,Hun Org Precision,Hun Org Recall,Hun Per F1,Hun Per Precision,Hun Per Recall,Eng Loss,Eng Overall Precision,Eng Overall Recall,Eng Overall F1,Eng Overall Accuracy,Eng Loc F1,Eng Loc Precision,Eng Loc Recall,Eng Misc F1,Eng Misc Precision,Eng Misc Recall,Eng Org F1,Eng Org Precision,Eng Org Recall,Eng Per F1,Eng Per Precision,Eng Per Recall,Ger Loss,Ger Overall Precision,Ger Overall Recall,Ger Overall F1,Ger Overall Accuracy,Ger Loc F1,Ger Loc Precision,Ger Loc Recall,Ger Misc F1,Ger Misc Precision,Ger Misc Recall,Ger Org F1,Ger Org Precision,Ger Org Recall,Ger Per F1,Ger Per Precision,Ger Per Recall
1,0.434899,No log,0.058244,0.803547,0.800634,0.802088,0.976120,0.760379,0.754335,0.766520,0.719033,0.862319,0.616580,0.784573,0.803030,0.766946,0.861111,0.840937,0.882277,0.019276,0.931919,0.930274,0.931095,0.994788,0.918239,0.893878,0.943966,0.806780,0.881481,0.743750,0.955123,0.946309,0.964103,0.977492,0.980645,0.974359,0.135293,0.624319,0.598261,0.611012,0.950872,0.392617,0.416370,0.371429,0.000000,0.000000,0.000000,0.168142,0.223529,0.134752,0.790265,0.750000,0.835098,0.054130,0.828017,0.840450,0.834187,0.983291,0.846384,0.825175,0.868712,0.000000,0.000000,0.000000,0.730570,0.740806,0.720613,0.904110,0.902873,0.905350
2,0.084267,No log,0.055650,0.789662,0.851382,0.819361,0.976295,0.798729,0.769388,0.830396,0.762178,0.852564,0.689119,0.795044,0.742725,0.855293,0.865804,0.845774,0.886805,0.016085,0.928387,0.949691,0.938918,0.996031,0.921776,0.904564,0.939655,0.863636,0.898649,0.831250,0.959064,0.937908,0.981197,0.961783,0.955696,0.967949,0.132372,0.613419,0.667826,0.639467,0.950410,0.531387,0.491892,0.577778,0.000000,0.000000,0.000000,0.256944,0.251701,0.262411,0.790497,0.754121,0.830560,0.051627,0.816205,0.898170,0.855228,0.983411,0.873357,0.850990,0.896933,0.000000,0.000000,0.000000,0.765894,0.682667,0.872232,0.916667,0.912925,0.920439
3,0.074450,No log,0.051338,0.829802,0.833937,0.831864,0.978389,0.810033,0.813936,0.806167,0.783562,0.831395,0.740933,0.821429,0.819560,0.823305,0.865054,0.851504,0.879043,0.015058,0.938315,0.953222,0.945709,0.996311,0.930526,0.909465,0.952586,0.873846,0.860606,0.887500,0.967687,0.962775,0.972650,0.961039,0.973684,0.948718,0.131854,0.662890,0.610435,0.635582,0.953178,0.484642,0.523985,0.450794,0.050000,0.142857,0.030303,0.189189,0.259259,0.148936,0.790595,0.768571,0.813918,0.045030,0.852268,0.890662,0.871042,0.986003,0.890909,0.880240,0.901840,No Log,No Log,No Log,0.793522,0.756173,0.834753,0.913781,0.904570,0.923182
4,0.064491,No log,0.052429,0.815600,0.845718,0.830386,0.977843,0.805316,0.788326,0.823054,0.765499,0.797753,0.735751,0.820801,0.806618,0.835491,0.868163,0.849103,0.888098,0.015445,0.938261,0.952339,0.945247,0.996151,0.926004,0.908714,0.943966,0.878505,0.875776,0.881250,0.968617,0.961279,0.976068,0.954839,0.961039,0.948718,0.132443,0.633535,0.637391,0.635457,0.952091,0.494624,0.479167,0.511111,0.040000,0.058824,0.030303,0.203390,0.252632,0.170213,0.798540,0.771509,0.827534,0.046540,0.845747,0.900516,0.872273,0.985595,0.892771,0.876923,0.909202,No Log,No Log,No Log,0.795545,0.746269,0.851789,0.914363,0.899204,0.930041
5,0.061699,No log,0.053185,0.814507,0.844585,0.829274,0.977544,0.804781,0.794139,0.815712,0.757895,0.770053,0.746114,0.819429,0.798410,0.841584,0.867745,0.851276,0.884864,0.015580,0.939077,0.952339,0.945662,0.996151,0.926004,0.908714,0.943966,0.878505,0.875776,0.881250,0.968617,0.961279,0.976068,0.957929,0.967

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

eval/combined_LOC_f1,▁▆█▇▇
eval/combined_LOC_precision,▁▃█▅▆
eval/combined_LOC_recall,▁█▅▇▆
eval/combined_MISC_f1,▁▆█▆▅
eval/combined_MISC_precision,█▇▆▃▁
eval/combined_MISC_recall,▁▅█▇█
eval/combined_ORG_f1,▁▃███
eval/combined_ORG_precision,▆▁█▇▆
eval/combined_ORG_recall,▁█▅▆▇
eval/combined_PER_f1,▁▆▅██
+75,...


## Final Test Set Evaluation

Performing a comprehensive evaluation across separate Hungarian, English, and German test sets to assess cross-lingual generalization.

In [18]:
test_datasets = {
    "combined": tokenized_dataset["test"],
    "test_hun": tokenized_hun_dataset["test"],
    "test_eng": tokenized_eng_dataset["test"],
    "test_ger": tokenized_ger_dataset["test"]
}

In [19]:
# Disabling all multiprocessing features for stable evaluation
trainer.args.dataloader_num_workers = 0
trainer.args.dataloader_persistent_workers = False
trainer.args.dataloader_prefetch_factor = None

# Fix 'on_train_begin' error by removing notebook callback
trainer.remove_callback(NotebookProgressCallback)

wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_eng_ger_test"
    )
trainer.args.dataloader_num_workers = 0
trainer.args.dataloader_persistent_workers = False
trainer.args.dataloader_prefetch_factor = None

test_results = trainer.evaluate(eval_dataset=test_datasets)

wandb.finish()


early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/combined_LOC_f1,▁
eval/combined_LOC_precision,▁
eval/combined_LOC_recall,▁
eval/combined_MISC_f1,▁
eval/combined_MISC_precision,▁
eval/combined_MISC_recall,▁
eval/combined_ORG_f1,▁
eval/combined_ORG_precision,▁
eval/combined_ORG_recall,▁
eval/combined_PER_f1,▁
+72,...


#### Inspecting testing results

Charts in Weights&Biases are clearly showing that the model performs significantly worse on the English dataset in all types of labels.

In [20]:
test_results.keys()

dict_keys(['eval_combined_loss', 'eval_combined_overall_precision', 'eval_combined_overall_recall', 'eval_combined_overall_f1', 'eval_combined_overall_accuracy', 'eval_combined_LOC_f1', 'eval_combined_LOC_precision', 'eval_combined_LOC_recall', 'eval_combined_MISC_f1', 'eval_combined_MISC_precision', 'eval_combined_MISC_recall', 'eval_combined_ORG_f1', 'eval_combined_ORG_precision', 'eval_combined_ORG_recall', 'eval_combined_PER_f1', 'eval_combined_PER_precision', 'eval_combined_PER_recall', 'eval_combined_runtime', 'eval_combined_samples_per_second', 'eval_combined_steps_per_second', 'epoch', 'eval_test_hun_loss', 'eval_test_hun_overall_precision', 'eval_test_hun_overall_recall', 'eval_test_hun_overall_f1', 'eval_test_hun_overall_accuracy', 'eval_test_hun_LOC_f1', 'eval_test_hun_LOC_precision', 'eval_test_hun_LOC_recall', 'eval_test_hun_MISC_f1', 'eval_test_hun_MISC_precision', 'eval_test_hun_MISC_recall', 'eval_test_hun_ORG_f1', 'eval_test_hun_ORG_precision', 'eval_test_hun_ORG_recal

In [21]:
# Most important metrics
print(f"Overall F1 score:               {test_results["eval_combined_overall_f1"]*100:.2f}%")
print(f"F1 score on Hungarian dataset:  {test_results["eval_test_hun_overall_f1"]*100:.2f}%")
print(f"F1 score on English dataset:    {test_results["eval_test_eng_overall_f1"]*100:.2f}%")
print(f"F1 score on German dataset:     {test_results["eval_test_ger_overall_f1"]*100:.2f}%")

Overall F1 score:               85.02%
F1 score on Hungarian dataset:  94.20%
F1 score on English dataset:    60.81%
F1 score on German dataset:     86.25%


Labels per category in the English dataset

In [22]:
print(f"F1 score on PER labels:     {test_results["eval_test_eng_PER_f1"]*100:.2f}%")
print(f"F1 score on LOC labels:     {test_results["eval_test_eng_LOC_f1"]*100:.2f}%")
print(f"F1 score on ORG labels:     {test_results["eval_test_eng_ORG_f1"]*100:.2f}%")
print(f"F1 score on MISC labels:    {test_results["eval_test_eng_MISC_f1"]*100:.2f}%")


F1 score on PER labels:     76.33%
F1 score on LOC labels:     46.89%
F1 score on ORG labels:     19.38%
F1 score on MISC labels:    0.00%


Since the English dataset performance on all labels is weak, we might suspect that it acts as noise in the model. This is could be attributed to its quality (silver).

The performance on the English dataset is so weak, that is doesn't worth spending time on other possible label-mappings, we should rather focus on defining clear boundaries for Hungarian and German language for the model by training it on the other two datasets.

## Fine tuning on Gold-only dataset

Creating labels from gold_only dataset for compute metrics function

In [23]:
gold_only_label_names = gold_only_dataset['train']['ner'].features.feature.names


In [24]:
gold_only_label_names

['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']

In [25]:
args = TrainingArguments(
    output_dir="../xlm-roberta-ner-hun_ger",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    metric_for_best_model="eval_combined_loss",
    load_best_model_at_end=True,
    greater_is_better=False,
    per_device_train_batch_size=96,
    per_device_eval_batch_size=192,
    bf16=True,
    bf16_full_eval=True,
    dataloader_num_workers=8,
    dataloader_persistent_workers=True,
    dataloader_prefetch_factor=2,
    learning_rate=1e-5,
    num_train_epochs=5,
    lr_scheduler_type="cosine",
    weight_decay=0.01,
    warmup_ratio=0.1,
    push_to_hub=False,
    report_to="wandb",
    run_name="ner_exp_hun_ger"
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [26]:
eval_datasets = {
    "combined": gold_only_tokenized_dataset["validation"],
    "hun": tokenized_hun_dataset["validation"],
    "ger": tokenized_ger_dataset["validation"]
}

In [27]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=gold_only_tokenized_dataset["train"],
    eval_dataset=eval_datasets,
    data_collator=data_collator,
    compute_metrics=lambda x: compute_metrics(x, label_names=gold_only_label_names),
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [28]:
wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_ger_training"
    )

trainer.train()

wandb.finish()

Epoch,Training Loss,Validation Loss,Combined Loss,Combined Overall Precision,Combined Overall Recall,Combined Overall F1,Combined Overall Accuracy,Combined Loc F1,Combined Loc Precision,Combined Loc Recall,Combined Misc F1,Combined Misc Precision,Combined Misc Recall,Combined Org F1,Combined Org Precision,Combined Org Recall,Combined Per F1,Combined Per Precision,Combined Per Recall,Hun Loss,Hun Overall Precision,Hun Overall Recall,Hun Overall F1,Hun Overall Accuracy,Hun Loc F1,Hun Loc Precision,Hun Loc Recall,Hun Misc F1,Hun Misc Precision,Hun Misc Recall,Hun Org F1,Hun Org Precision,Hun Org Recall,Hun Per F1,Hun Per Precision,Hun Per Recall,Ger Loss,Ger Overall Precision,Ger Overall Recall,Ger Overall F1,Ger Overall Accuracy,Ger Loc F1,Ger Loc Precision,Ger Loc Recall,Ger Org F1,Ger Org Precision,Ger Org Recall,Ger Per F1,Ger Per Precision,Ger Per Recall
1,0.021922,No log,0.037416,0.872960,0.917892,0.894863,0.989368,0.898876,0.881543,0.916905,0.878505,0.875776,0.881250,0.873377,0.832817,0.918089,0.922817,0.920225,0.925424,0.015197,0.939130,0.953222,0.946124,0.996191,0.928270,0.909091,0.948276,0.878505,0.875776,0.881250,0.967797,0.959664,0.976068,0.961039,0.973684,0.948718,0.046952,0.839614,0.899108,0.868344,0.985283,0.890493,0.873672,0.907975,0.786604,0.724534,0.860307,0.914792,0.909214,0.920439
2,0.021360,No log,0.036745,0.874561,0.916360,0.894973,0.989474,0.898592,0.883657,0.914040,0.872274,0.869565,0.875000,0.874745,0.838155,0.914676,0.922559,0.916388,0.928814,0.015204,0.937446,0.952339,0.944834,0.996231,0.926316,0.905350,0.948276,0.872274,0.869565,0.875000,0.967797,0.959664,0.976068,0.961039,0.973684,0.948718,0.045961,0.842221,0.896762,0.868636,0.985403,0.890634,0.877381,0.904294,0.786782,0.730994,0.851789,0.914518,0.904698,0.924554
3,0.021193,No log,0.036973,0.873141,0.917279,0.894666,0.989534,0.898687,0.882949,0.914995,0.869565,0.864198,0.875000,0.874695,0.835925,0.917235,0.921954,0.916295,0.927684,0.015271,0.937391,0.951456,0.944371,0.996111,0.928270,0.909091,0.948276,0.869565,0.864198,0.875000,0.968617,0.961279,0.976068,0.954545,0.967105,0.942308,0.046271,0.841089,0.899108,0.869131,0.985595,0.890229,0.875445,0.905521,0.788732,0.729378,0.858603,0.915139,0.905914,0.924554
4,0.021202,No log,0.036948,0.874197,0.917586,0.895366,0.989579,0.899014,0.884473,0.914040,0.872274,0.869565,0.875000,0.875865,0.837354,0.918089,0.922042,0.915367,0.928814,0.015244,0.938207,0.951456,0.944785,0.996151,0.928270,0.909091,0.948276,0.872274,0.869565,0.875000,0.968617,0.961279,0.976068,0.954545,0.967105,0.942308,0.046260,0.841897,0.899578,0.869782,0.985643,0.890634,0.877381,0.904294,0.790297,0.730825,0.860307,0.915254,0.904826,0.925926


early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

eval/combined_LOC_f1,▆▁▃█
eval/combined_LOC_precision,▁▆▄█
eval/combined_LOC_recall,█▁▃▁
eval/combined_MISC_f1,█▃▁▃
eval/combined_MISC_precision,█▄▁▄
eval/combined_MISC_recall,█▁▁▁
eval/combined_ORG_f1,▁▅▅█
eval/combined_ORG_precision,▁█▅▇
eval/combined_ORG_recall,█▁▆█
eval/combined_PER_f1,█▆▁▂
+52,...


In [29]:
test_datasets = {
    "combined": gold_only_tokenized_dataset["test"],
    "test_hun": tokenized_hun_dataset["test"],
    "test_ger": tokenized_ger_dataset["test"]
}

In [30]:
# Disabling all multiprocessing features for stable evaluation
trainer.args.dataloader_num_workers = 0
trainer.args.dataloader_persistent_workers = False
trainer.args.dataloader_prefetch_factor = None

# Fix 'on_train_begin' error by removing notebook callback
trainer.remove_callback(NotebookProgressCallback)

wandb.init(
    entity="rchrdhlzhfr-personal",
    project="polyglot-ner-project",
    name="hun_ger_test"
    )

test_results = trainer.evaluate(eval_dataset=test_datasets)

wandb.finish()


early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
d:\Projects\polyglot_ner_project\.venv\Lib\site-packages\seqeval\metrics\v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
early stopping required metric_for_best_model, but did not find eval_combined_loss so early stopping is disabled
wandb: ERROR The nbformat package was not found. It is required to save notebook history.


eval/combined_LOC_f1,▁
eval/combined_LOC_precision,▁
eval/combined_LOC_recall,▁
eval/combined_MISC_f1,▁
eval/combined_MISC_precision,▁
eval/combined_MISC_recall,▁
eval/combined_ORG_f1,▁
eval/combined_ORG_precision,▁
eval/combined_ORG_recall,▁
eval/combined_PER_f1,▁
+52,...


## Inspecting testing results on gold-only datasets

In [35]:
# Most important metrics
print(f"Overall F1 score:               {test_results["eval_combined_overall_f1"]*100:.2f}%")
print(f"F1 score on Hungarian dataset:  {test_results["eval_test_hun_overall_f1"]*100:.2f}%")
print(f"F1 score on German dataset:     {test_results["eval_test_ger_overall_f1"]*100:.2f}%")

Overall F1 score:               89.75%
F1 score on Hungarian dataset:  96.47%
F1 score on German dataset:     85.64%


Labels per category in the Hungarian dataset

In [37]:
# Hungarian
print(f"F1 score on PER labels:     {test_results["eval_test_hun_PER_f1"]*100:.2f}%")
print(f"F1 score on LOC labels:     {test_results["eval_test_hun_LOC_f1"]*100:.2f}%")
print(f"F1 score on ORG labels:     {test_results["eval_test_hun_ORG_f1"]*100:.2f}%")
print(f"F1 score on MISC labels:    {test_results["eval_test_hun_MISC_f1"]*100:.2f}%")

F1 score on PER labels:     96.55%
F1 score on LOC labels:     95.67%
F1 score on ORG labels:     98.40%
F1 score on MISC labels:    85.49%


Labels per category in the German dataset

In [38]:
# German
print(f"F1 score on PER labels:     {test_results["eval_test_ger_PER_f1"]*100:.2f}%")
print(f"F1 score on LOC labels:     {test_results["eval_test_ger_LOC_f1"]*100:.2f}%")
print(f"F1 score on ORG labels:     {test_results["eval_test_ger_ORG_f1"]*100:.2f}%")
print(f"F1 score on MISC labels:    {test_results["eval_test_ger_MISC_f1"]*100:.2f}%")

F1 score on PER labels:     90.33%
F1 score on LOC labels:     87.83%
F1 score on ORG labels:     76.83%
F1 score on MISC labels:    0.00%


By removing the English dataset we could prove that the model (xlm-roberta-base) is fully capable of learning the task, yielding ~96% on Hungarian and ~85% on German language.

If we include the English dataset, the end users will experience terrible performance on English text. This destroys trust in the model. So I would recommend sending the Hun-Ger NER model to production and getting a better quality English dataset to be able to get decent performance (85+%) on all languages.

## Saving the model and pushing to HuggingFace Hub

In [39]:
# Creating folder if it does not exist

os.makedirs("../models", exist_ok=True)

save_directory = "../models/xlm-roberta-ner-hun_ger"
# Saving the model
trainer.save_model(save_directory)

# Saving the tokenizer
tokenizer.save_pretrained(save_directory)

print(f"Model and tokenizer successfully saved to {save_directory}")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model and tokenizer successfully saved to ../models/xlm-roberta-ner-hun_ger


In [41]:
# Pushing model to HuggingFace Hub
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

CommitInfo(commit_url='https://huggingface.co/RichardHolzhofer/xlm-roberta-ner-hun_ger/commit/c16c42f2557d81099ad835a3ea913ac6b0d7f69e', commit_message='End of training', commit_description='', oid='c16c42f2557d81099ad835a3ea913ac6b0d7f69e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/RichardHolzhofer/xlm-roberta-ner-hun_ger', endpoint='https://huggingface.co', repo_type='model', repo_id='RichardHolzhofer/xlm-roberta-ner-hun_ger'), pr_revision=None, pr_num=None)

## Inferencing with the model via HuggingFace Hub

In [45]:
ner_pipeline = pipeline(
    "ner",
    model="RichardHolzhofer/xlm-roberta-ner-hun_ger",
    aggregation_strategy="simple"
    )

# Hungarian Example
# Translation: "János Kovács visited the OTP Bank headquarters in Budapest yesterday."
hun_text = "Kovács János tegnap az OTP Bank központjában, Budapesten járt."

print("--- Hungarian Prediction ---")
for entity in ner_pipeline(hun_text):
    print(f"Word: {entity['word']:<15} | Label: {entity['entity_group']:<5} | Score: {entity['score']:.4f}")
print("\n" + "="*50 + "\n")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

--- Hungarian Prediction ---
Word: Kovács János    | Label: PER   | Score: 0.9959
Word: OTP Bank        | Label: ORG   | Score: 0.9765
Word: Budapesten      | Label: LOC   | Score: 0.9962




In [46]:
# German Example
# Translation: "Angela Merkel visited the BMW headquarters in Munich yesterday."
ger_text = "Angela Merkel besuchte gestern die Zentrale von BMW in München."
print("--- German Prediction ---")
for entity in ner_pipeline(ger_text):
    print(f"Word: {entity['word']:<15} | Label: {entity['entity_group']:<5} | Score: {entity['score']:.4f}")

--- German Prediction ---
Word: Angela Merkel   | Label: PER   | Score: 0.9913
Word: BMW             | Label: ORG   | Score: 0.9971
Word: München         | Label: LOC   | Score: 0.9968


### 📊 [View Training Results on Weights & Biases Dashboard](https://wandb.ai/rchrdhlzhfr-personal/polyglot-ner-project/workspace?nw=nwuserrchrdhlzhfr)
